# MODULE 8: MODEL DEPLOYMENT

## Topics Covered
1. Pickle & Joblib (Saving the Model)
2. Flask / FastAPI (Web Frameworks)
3. REST API Creation
4. Docker Basics
5. Cloud Deployment Basics
6. CI/CD for ML

In [1]:
# Install required libraries (run once)
# !pip install scikit-learn joblib flask fastapi uvicorn pydantic httpx

import os, json, tempfile, datetime
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Tiny model we will reuse in every section
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=200).fit(X_tr, y_tr)

WORKDIR = tempfile.mkdtemp()
print(f'Test accuracy: {model.score(X_te, y_te):.3f}')
print(f'Working dir:   {WORKDIR}')
print('Setup done!')

Test accuracy: 1.000
Working dir:   /var/folders/11/scnp1vm97gx749r6nb9shbv9ymhsbd/T/tmpq21_78qu
Setup done!


---
# SECTION 1: PICKLE & JOBLIB

## What is Serialization? (Start Here)

**Simple Story:**
You spent 3 hours training a model in your notebook. You close your laptop.
Tomorrow you open it again — the model is **gone**. The Python object lived only in RAM.

**The fix — save it to a file:**
```
Trained model in RAM   ──►  bytes on disk (a .pkl file)   ──►  load tomorrow → same model
       ↑                            ↑                                  ↑
    Python object             Serialization                    Deserialization
```

## Definition
> **Serialization** is the process of converting a Python object (e.g. a trained model) into a stream of bytes that can be saved to disk or sent over a network.

> **Deserialization** is the reverse — reading those bytes back into a live Python object.

## Two Tools

> **Pickle:** Python's built-in serialization. Works on *any* Python object. Already installed.

> **Joblib:** External library, optimized for objects with **large NumPy arrays** (which is exactly what sklearn models are). Faster, smaller files for ML models.

| Feature | Pickle | Joblib |
|---------|--------|--------|
| Built-in? | Yes | No (comes with sklearn) |
| Best for | Any Python object | Sklearn models, NumPy-heavy objects |
| Speed (sklearn) | Slower | Faster |
| File size (sklearn) | Larger | Smaller (compressed) |
| API | `pickle.dump(obj, f)` | `joblib.dump(obj, path)` |

## Security Warning
Never load a `.pkl` / `.joblib` file from an untrusted source. A malicious file can execute arbitrary code when loaded.

In [2]:
# Pickle — the built-in way
import pickle

pkl_path = os.path.join(WORKDIR, 'model.pkl')

# ── SAVE ──
with open(pkl_path, 'wb') as f:        # 'wb' = write-binary
    pickle.dump(model, f)

# ── LOAD ──
with open(pkl_path, 'rb') as f:        # 'rb' = read-binary
    model_pkl = pickle.load(f)

# ── VERIFY (predictions identical) ──
print(f'File size:        {os.path.getsize(pkl_path)} bytes')
print(f'Predictions match: {np.array_equal(model.predict(X_te), model_pkl.predict(X_te))}')

File size:        828 bytes
Predictions match: True


In [ ]:
# 1. try saving json file in both pickle and joblib
# 2. try saving dataframe in both pickle and joblib
# 3. try saving json file in both pickle and joblib with compression

In [3]:
# Joblib — preferred for sklearn
import joblib

jl_path = os.path.join(WORKDIR, 'model.joblib')

joblib.dump(model, jl_path)             # save (one line — handles file open)
model_jl = joblib.load(jl_path)         # load

print(f'File size:        {os.path.getsize(jl_path)} bytes')
print(f'Predictions match: {np.array_equal(model.predict(X_te), model_jl.predict(X_te))}')

File size:        975 bytes
Predictions match: True


In [4]:
# In production, save metadata next to the model — version, date, feature names, score
metadata = {
    'model_version': '1.0.0',
    'trained_at':   datetime.datetime.now().isoformat(timespec='seconds'),
    'feature_names':['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid'],
    'classes':      ['setosa', 'versicolor', 'virginica'],
    'test_accuracy':round(model.score(X_te, y_te), 4),
}
meta_path = os.path.join(WORKDIR, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(open(meta_path).read())

{
  "model_version": "1.0.0",
  "trained_at": "2026-05-19T18:43:51",
  "feature_names": [
    "sepal_len",
    "sepal_wid",
    "petal_len",
    "petal_wid"
  ],
  "classes": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "test_accuracy": 1.0
}


| | |
|--|--|
| ✅ Saves trained model to disk | ❌ `.pkl` files only loadable in Python |
| ✅ Loads instantly (no retrain) | ❌ Tied to library versions (sklearn 1.2 ≠ 1.4) |
| ✅ Joblib compresses NumPy arrays | ❌ Security risk if loading untrusted files |

> **Reference:** [GeeksForGeeks — Save model with pickle/joblib](https://www.geeksforgeeks.org/saving-a-machine-learning-model/)

---
# SECTION 2: FLASK & FASTAPI (Web Frameworks)

## What is a Web Framework? (Start Here)

**Simple Story:**
Your model can predict in Python. But how does a *mobile app* or *another server* get a prediction?
Answer — wrap the model behind a **web framework** so it speaks HTTP.

```
Mobile app  ──HTTP request──►  Your Python web server  ──►  model.predict()  ──►  JSON back
(any language)                 (Flask or FastAPI)
```

## Definition
> **Web framework:** A library that lets your Python functions respond to HTTP requests. You write functions; the framework maps URLs to those functions.

## Sub-Definitions
> **Route / Endpoint:** A URL path bound to a Python function. `/predict` is a route.

> **Decorator (`@app.route`):** Tells the framework "call this function when someone hits this URL."

> **Request / Response:** The data the client sends in / what your function returns.

> **JSON:** The standard data format both ways. Python `dict` ↔ JSON string.

> **Test client:** A fake HTTP caller used in tests and notebooks. Calls your routes in-memory without starting a real server (which would block the kernel).

## Flask vs FastAPI

| Feature | Flask | FastAPI |
|---------|-------|---------|
| Style | Simple, sync | Modern, async |
| Validation | Manual (`if 'x' not in data`) | Automatic (Pydantic) |
| Auto-docs | No (write yourself) | Yes (Swagger UI at `/docs`) |
| Speed | Good | Excellent |
| Learning curve | Easiest | Slightly more |
| Use when | Tiny services, legacy | New projects, public APIs |

## The Test Client Trick
```
Real production:               app.run(port=5000)   ← blocks forever, listens on network
Inside this notebook:          app.test_client()    ← fake calls, runs and exits
```

In [5]:
# FLASK — minimal API with two routes
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok'})

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()                                    # parse JSON body
    feats = np.array(data['features']).reshape(1, -1)
    pred = int(model.predict(feats)[0])
    return jsonify({'class': pred, 'name': metadata['classes'][pred]})

# ── Simulate HTTP calls (no real server) ──
with app.test_client() as client:
    r1 = client.get('/health')
    print(f'GET  /health   → status {r1.status_code}, body {r1.get_json()}')

    r2 = client.post('/predict', json={'features': X_te[0].tolist()})
    print(f'POST /predict  → status {r2.status_code}, body {r2.get_json()}')

GET  /health   → status 200, body {'status': 'ok'}
POST /predict  → status 200, body {'class': 1, 'name': 'versicolor'}


---
# SECTION 3: REST API CREATION

## What is REST? (Start Here)

**Simple Story:**
You and a friend agree on rules for talking on the phone — you greet, you take turns, you say bye.
REST is the same idea for *programs* talking over HTTP. It is a set of **conventions**, not a library.

An API is "RESTful" if it follows these conventions.

## Definition
> **REST (Representational State Transfer):** A style for designing web APIs over HTTP, using URLs to identify resources and HTTP verbs to describe actions.

## The 4 Core Ideas

> **1. Resources:** Every "thing" your API exposes lives at a URL.
> ```
> /models/iris       → the iris model
> /users/42          → user with id 42
> /predictions       → the predictions collection
> ```

> **2. HTTP verbs describe the action:**

| Verb | Meaning | Example |
|------|---------|---------|
| `GET` | Read (no side effects) | `GET /version` |
| `POST` | Create / submit | `POST /predict` |
| `PUT` | Replace | `PUT /models/iris` |
| `PATCH` | Partial update | `PATCH /users/42` |
| `DELETE` | Remove | `DELETE /models/iris` |

> **3. Stateless:** Every request must contain everything needed to handle it. The server keeps no per-client memory.

> **4. JSON:** The standard data format. Request body is JSON; response body is JSON.

## Status Codes Carry Meaning
Don't return `{"error": "..."}` with status 200 — load balancers, monitoring, retries all read the status code.

| Code | Meaning |
|------|---------|
| 200  | OK |
| 201  | Created |
| 400  | Bad Request — client sent garbage |
| 401  | Unauthorized |
| 404  | Not Found |
| 422  | Unprocessable Entity — validation failed |
| 500  | Server Error — your code crashed |

## A Typical ML REST API
```
GET    /health           → liveness probe (used by Kubernetes / load balancers)
GET    /version          → which model is loaded
POST   /predict          → inference for one row
POST   /predict/batch    → inference for many rows in one call
```

In [ ]:
# A REST API following the 4 conventions: resources · verbs · stateless · JSON
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

rest = FastAPI()

class OneRow(BaseModel):
    features: List[float]

class ManyRows(BaseModel):
    rows: List[List[float]]

@rest.get('/health')                           # GET — read-only probe
def health():
    return {'status': 'ok'}

@rest.get('/version')                          # GET — model metadata
def version():
    return {'model_version': metadata['model_version'], 'trained_at': metadata['trained_at']}

@rest.post('/predict')                         # POST — submit data, get result
def predict_one(req: OneRow):
    if len(req.features) != 4:
        raise HTTPException(status_code=400, detail='Iris model expects exactly 4 features')
    pred = int(model.predict(np.array(req.features).reshape(1, -1))[0])
    return {'class': pred, 'name': metadata['classes'][pred]}

@rest.post('/predict/batch')                   # POST — bulk inference
def predict_batch(req: ManyRows):
    preds = model.predict(np.array(req.rows)).tolist()
    return {'predictions': preds, 'count': len(preds)}

c = TestClient(rest)
print('GET  /health         →', c.get('/health').json())
print('GET  /version        →', c.get('/version').json())
print('POST /predict        →', c.post('/predict', json={'features': X_te[0].tolist()}).json())
print('POST /predict/batch  →', c.post('/predict/batch', json={'rows': X_te[:3].tolist()}).json())

# ── Status code demo: bad request ──
bad = c.post('/predict', json={'features': [1.0, 2.0]})
print(f'\nBad request → status {bad.status_code}  (400 = client error)  body: {bad.json()}')

| | |
|--|--|
| ✅ Universal — every language can call it | ❌ Stateless = client must resend context every time |
| ✅ Verbs + status codes = self-documenting | ❌ JSON over HTTP is verbose vs binary protocols |
| ✅ Cacheable, observable, debuggable | ❌ Slower than gRPC for high-throughput internal calls |

> **Reference:** [GeeksForGeeks — REST API Architectural Constraints](https://www.geeksforgeeks.org/rest-api-architectural-constraints/)

---
# SECTION 4: DOCKER — DETAILED

## What is Docker? (Start Here)

**Simple Story:**
Your model works on your laptop (Python 3.11, sklearn 1.4). You email it to a colleague. It crashes on their machine (Python 3.9, sklearn 1.2). You deploy it to a server. It crashes again (missing system library `libgomp`). This is the classic **"works on my machine"** problem.

**The fix — ship the *whole environment* as one box:**
```
Without Docker:                          With Docker:
  app.py        → server                   ┌─────────────────────────────┐
  model.joblib                             │  app.py                     │
  needs Python 3.11                        │  model.joblib               │
  needs sklearn 1.4                        │  Python 3.11                │
  needs system libs                        │  sklearn 1.4                │  ──►  one image,
  ↓                                        │  system libs (libgomp...)   │       runs anywhere
  hope server has all this                 └─────────────────────────────┘
```

## Definition
> **Docker:** A tool that packages an application together with **its entire environment** (OS libraries, Python interpreter, Python packages, your code, your model file) into a single portable artifact called an **image**. Anyone with Docker can run the image identically on Linux, macOS, Windows, or any cloud — no installation steps, no version mismatches.

## Core Vocabulary — Carefully Defined

> **Image:** The recipe + frozen ingredients. A read-only snapshot of a filesystem (OS libs, Python, your code, your model). Built once, shared, never changes. Think of it like a **class**.

> **Container:** A *running instance* of an image. You can start, stop, and delete containers without touching the image. Many containers can run from the same image. Think of it like an **object** (instance of a class).

> **Dockerfile:** A plain text file (literally named `Dockerfile`, no extension) with the build instructions. Each line is one *step*: which base OS to start from, which packages to install, which files to copy, which command to run when the container starts.

> **Layer:** Each line in a Dockerfile creates a **layer** — a frozen filesystem diff. Docker caches layers. If you change only `app.py`, layers above the COPY of `app.py` are reused → builds are blazing fast.

> **Registry:** A server that stores images so others can pull them. Docker Hub is the public default. Private options: AWS ECR, Google GCR, GitHub Container Registry.

> **Tag:** A label for a specific version of an image. `iris-api:1.0` means image `iris-api` version `1.0`. `iris-api:latest` is the convention for "newest", but pin real versions in production.

> **Port mapping (`-p host:container`):** Containers have their own private network. To reach a port inside, you map it: `-p 8000:8000` means "host port 8000 → container port 8000".

> **Volume:** A folder shared between host and container. Used for data that should survive container restarts (databases, logs, training data).

## Container vs Virtual Machine

| | Virtual Machine | Container |
|--|----------------|-----------|
| Boots a full OS? | Yes — full Linux kernel | No — shares host kernel |
| Size | GBs (whole OS) | MBs (just libs + your app) |
| Startup | Minutes | < 1 second |
| Isolation | Strong (hardware-level) | Strong (process + namespace) |
| Density on one host | Tens | Hundreds to thousands |

A container is *not* a tiny VM — it's a **process** running on the host kernel inside an isolated namespace. That's why it starts in milliseconds.

## Anatomy of a Dockerfile (Line by Line)

```dockerfile
FROM python:3.11-slim
```
> Start from an existing image. `python:3.11-slim` = official Debian image with Python 3.11 pre-installed, stripped of non-essentials (~120 MB instead of ~1 GB for full `python:3.11`).

```dockerfile
WORKDIR /app
```
> Set the working directory inside the container. Equivalent to `mkdir -p /app && cd /app`. Every subsequent COPY/RUN/CMD runs relative to this.

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
```
> Copy `requirements.txt` from your host into `/app/` inside the image, then run pip install. We do this **before** copying the source code so this expensive layer is *cached* — re-runs are skipped when only the code changes.

```dockerfile
COPY main.py model.joblib ./
```
> Copy app code + saved model. Put this *after* dependency install so editing code doesn't invalidate the slow pip cache.

```dockerfile
EXPOSE 8000
```
> Documentation only — declares "this image listens on 8000". Doesn't actually open the port. You still need `-p 8000:8000` when running.

```dockerfile
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```
> The default command. `--host 0.0.0.0` is **critical** — by default uvicorn binds to `127.0.0.1` which is *inside* the container only. `0.0.0.0` means "listen on all interfaces" so the host can reach it.

## Layer Caching — The Performance Trick

Docker hashes each instruction's inputs. If the inputs haven't changed, it reuses the cached layer. **Order your Dockerfile from least-changing to most-changing:**

```
✅ Good ordering                  ❌ Bad ordering
1. FROM python:3.11-slim           1. FROM python:3.11-slim
2. WORKDIR /app                    2. WORKDIR /app
3. COPY requirements.txt           3. COPY . .                  ← any code edit
4. RUN pip install                 4. RUN pip install -r req...   busts cache
5. COPY app code                                               → reinstalls every build
6. CMD ...
```

## The 7 Commands You'll Use 90% of the Time

```
docker build -t myapp:1.0 .          ← build image from Dockerfile (`.` = current dir)
docker images                        ← list local images
docker run  -p 8000:8000 myapp:1.0   ← run container, map host port → container port
docker ps                            ← list running containers
docker logs <id>                     ← view container's stdout/stderr
docker stop <id>                     ← stop a running container
docker rm   <id>                     ← delete a stopped container
```

Common flags on `docker run`:
- `-d` — detached (run in background)
- `--name foo` — give the container a friendly name
- `-p 8000:8000` — port mapping
- `-e VAR=value` — environment variable
- `--rm` — auto-delete container when it stops

## Worked Example: Iris API in Docker (We'll Actually Run It Below)

The cells below will:
1. Write `main.py`, `requirements.txt`, `Dockerfile`, the saved model — into a real folder.
2. Run `docker build` to produce an image.
3. Run `docker run` to start the container.
4. Use `curl` (via Python `requests`) to send a real HTTP request and get a prediction back.
5. Stop and clean up the container.

> **Prerequisite:** Docker Desktop must be installed and running. Check with `docker --version` in a terminal. If it's not installed: https://docs.docker.com/get-docker/</cell id="cell-s4">

In [12]:
# A complete Dockerfile for our FastAPI iris service — line-by-line
dockerfile = '''
# Dockerfile
FROM python:3.11-slim                  # 1. start from a small Python base image

WORKDIR /app                           # 2. work inside /app inside the container

COPY requirements.txt .                # 3. copy deps file FIRST (Docker caches this layer)
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py model.joblib ./           # 4. now copy app code + saved model

EXPOSE 8000                            # 5. document the port (informational)

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements = '''
fastapi==0.110.0
uvicorn==0.29.0
scikit-learn==1.4.0
joblib==1.3.2
numpy==1.26.4
'''

dockerignore = '''
.git
__pycache__
*.ipynb
.venv
tests/
'''

for name, content in [('Dockerfile', dockerfile),
                       ('requirements.txt', requirements),
                       ('.dockerignore', dockerignore)]:
    print(f'===== {name} =====')
    print(content)

===== Dockerfile =====

# Dockerfile
FROM python:3.11-slim                  # 1. start from a small Python base image

WORKDIR /app                           # 2. work inside /app inside the container

COPY requirements.txt .                # 3. copy deps file FIRST (Docker caches this layer)
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py model.joblib ./           # 4. now copy app code + saved model

EXPOSE 8000                            # 5. document the port (informational)

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

===== requirements.txt =====

fastapi==0.110.0
uvicorn==0.29.0
scikit-learn==1.4.0
joblib==1.3.2
numpy==1.26.4

===== .dockerignore =====

.git
__pycache__
*.ipynb
.venv
tests/



In [13]:
# Typical command sequence — run these in a terminal once Docker is installed
commands = [
    ('Build image',     'docker build -t iris-api:1.0 .'),
    ('Run container',   'docker run -d --name iris -p 8000:8000 iris-api:1.0'),
    ('Send request',    'curl -X POST http://localhost:8000/predict '
                        '-H "Content-Type: application/json" '
                        '-d \'{"features":[5.1,3.5,1.4,0.2]}\''),
    ('Tail logs',       'docker logs -f iris'),
    ('Stop',            'docker stop iris'),
    ('Remove',          'docker rm iris'),
    ('Push to registry','docker push myregistry.com/iris-api:1.0'),
]
for desc, cmd in commands:
    print(f'# {desc}')
    print(f'  $ {cmd}\n')

# Build image
  $ docker build -t iris-api:1.0 .

# Run container
  $ docker run -d --name iris -p 8000:8000 iris-api:1.0

# Send request
  $ curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d '{"features":[5.1,3.5,1.4,0.2]}'

# Tail logs
  $ docker logs -f iris

# Stop
  $ docker stop iris

# Remove
  $ docker rm iris

# Push to registry
  $ docker push myregistry.com/iris-api:1.0



## ▶️ Run It Yourself — Full Docker Walkthrough

The next cells write all required files into a folder, then build and run a real Docker container with our iris model behind FastAPI. Run them top-to-bottom.

> **Need Docker?** macOS: `brew install --cask docker` then open Docker Desktop. Verify with `docker --version`. The build step downloads ~150 MB the first time (cached after).

| | |
|--|--|
| ✅ "Works on my machine" → "works everywhere" | ❌ Extra layer to learn and debug |
| ✅ Lightweight (MBs, seconds to start) | ❌ Image bloat if not careful (use `slim` bases) |
| ✅ Pinned dependencies → reproducibility | ❌ Linux containers can't run on Windows kernel directly |

> **Reference:** [GeeksForGeeks — Docker Tutorial](https://www.geeksforgeeks.org/docker-tutorial/) · [Dockerize a Python ML app](https://www.geeksforgeeks.org/dockerizing-a-simple-python-application/)

---
# SECTION 5: CLOUD DEPLOYMENT BASICS — DETAILED

## What Does "Deploy to the Cloud" Mean? (Start Here)

**Simple Story:**
Your Docker image runs perfectly on your laptop. But your laptop:
1. Sleeps when you close the lid,
2. Has a private IP no one on the internet can reach,
3. Has 1 CPU — it'll choke if 100 users hit `/predict` at once,
4. Will catch fire if you try to host a real product on it.

You need a computer that is **always on, internet-reachable, and elastic**. Renting one is what *cloud deployment* means.

## Definition

> **Cloud:** Someone else's computers — owned by AWS, Google, Microsoft, etc. — that you rent by the hour, the request, or the GB. You don't buy hardware; you call an API and a server appears in 30 seconds.

> **Deployment:** The act of moving your built artifact (Docker image, zip file, code repo) from your laptop to a cloud computer that serves real traffic at a public URL.

## The 3 Big Cloud Providers

| Provider | Brand | Strengths |
|----------|-------|-----------|
| **AWS** (Amazon Web Services) | the market leader | most services, biggest community, hardest UI |
| **GCP** (Google Cloud Platform) | the data/ML one | best for ML (Vertex AI, BigQuery), cleanest UX |
| **Azure** (Microsoft) | the enterprise one | best AD/Office365 integration, many large companies |

For ML hobby projects, GCP is often friendliest. For enterprise jobs, you'll mostly see AWS.

## Service-Model Vocabulary (Acronyms Decoded)

> **IaaS — Infrastructure as a Service:** You rent a *raw machine* (CPU, RAM, disk). You install everything else. Example: AWS EC2.

> **CaaS — Containers as a Service:** You hand the cloud a Docker image. It runs containers for you. Example: AWS Fargate, Google Cloud Run.

> **PaaS — Platform as a Service:** You hand the cloud your *code* (a Git repo). It builds + runs it. Example: Heroku, Render.

> **FaaS — Functions as a Service / Serverless:** You hand the cloud a *single function*. It runs only when called. Example: AWS Lambda, Cloud Functions.

> **MLaaS — ML as a Service:** Purpose-built ML platforms. Example: SageMaker, Vertex AI, Azure ML.

```
You provide:    Hardware    OS    Runtime    Code    Function
                ──────────────────────────────────────────────►
On-prem:        you         you   you        you     you
IaaS (EC2):     cloud       you   you        you     you
CaaS (Run):     cloud       cloud cloud      you     you
PaaS (Render):  cloud       cloud cloud      you     you
FaaS (Lambda):  cloud       cloud cloud      cloud   you
SaaS (gmail):   cloud       cloud cloud      cloud   cloud
```
The further down the list, the *less* you operate, the *more* the vendor charges per unit, and the *more* lock-in you accept.

## The 5 Flavours for Hosting an ML API — Most Control → Least Work

```
Most control                                                                Least work
    │                                                                            │
    ▼                                                                            ▼
  ┌──────┐    ┌──────────────┐    ┌──────────┐    ┌────────┐    ┌──────────────────┐
  │  VM  │    │ K8s / ECS    │    │ Cloud Run│    │ Lambda │    │ SageMaker /      │
  │      │    │ (orchestr.)  │    │ Fargate  │    │  (FaaS)│    │ Vertex AI        │
  └──────┘    └──────────────┘    └──────────┘    └────────┘    └──────────────────┘
  raw box     containers + auto    serverless     just code     managed ML platform
  you patch   restart + scale      scale-to-zero  zip files     auto-scale + monitor
```

## Definitions of Each Option (with Concrete Examples)

> **1️⃣ Virtual Machine (AWS EC2, GCP Compute Engine, Azure VM):** A rented Linux box. You SSH in (`ssh ubuntu@1.2.3.4`) and run Docker yourself.
> ✅ Total control, predictable cost, no cold starts, can install anything.
> ❌ You manage OS patches, restarts, scaling, security updates. One box → one point of failure.
> 💲 ~$5–$50/month for small instances, billed per-hour even when idle.

> **2️⃣ Container Orchestration (Kubernetes / EKS, AWS ECS):** You give it your image and a desired count ("run 5 copies"). It places them across machines, restarts crashed ones, rolls out new versions without downtime.
> ✅ Real production load, rolling deploys, self-healing, multi-region.
> ❌ Steep learning curve. K8s has its own vocabulary (pods, services, ingresses, deployments). Overkill for one tiny model.
> 💲 ~$70/month minimum for the control plane + the cost of nodes.

> **3️⃣ Serverless Containers (Google Cloud Run, AWS Fargate + ALB):** Push image → public HTTPS URL appears. Auto-scales 0 → many. Pay per request.
> ✅ Near-zero ops, scales to zero (free when idle), HTTPS for free, deploy in 60 s.
> ❌ Cold-start latency on first request after idle (1–5 s for ML images). Memory caps (~8 GB on Cloud Run).
> 💲 Free tier covers most demos. Then cents per million requests.
> 🎯 **Best default for an ML API in 2026.**

> **4️⃣ Function-as-a-Service (AWS Lambda, GCP Cloud Functions, Azure Functions):** Upload just a Python function. No server concept at all.
> ✅ Cheapest for occasional calls, instant scale.
> ❌ 250 MB unzipped code limit (Lambda) — sklearn + numpy + your model often don't fit. 15 min max execution. Cold starts. Hard to debug.
> 💲 Free tier = 1M requests/month.
> 🎯 Good for **light** ML (small XGBoost models, post-processing). Bad for big PyTorch models.

> **5️⃣ Managed ML Platform (AWS SageMaker, GCP Vertex AI, Azure ML):** Purpose-built for ML — training, hyperparameter tuning, model registry, hosted endpoints, drift monitoring, A/B testing all in one.
> ✅ Auto-scaling endpoints, A/B model versions, drift detection, GPU-friendly, integrates with cloud data warehouse.
> ❌ Heavy vendor lock-in, expensive at small scale, learning curve on each platform's quirks.
> 💲 ~$50/month minimum for a small always-on endpoint.
> 🎯 Right when you have **multiple models in production** and need governance.

> **6️⃣ Quick-Demo Platforms (Hugging Face Spaces, Streamlit Cloud, Render, Railway):** Push a Git repo → live URL. Free tiers.
> ✅ Perfect for portfolios, MVPs, sharing with stakeholders. No cloud account needed.
> ❌ Limited resources, slow cold starts, not for real production traffic.
> 💲 Free for demos.
> 🎯 Use this for **the project on your résumé**.

## 10-Second Decision Guide

| Situation | Pick |
|-----------|------|
| Hackathon / portfolio demo | **Hugging Face Spaces** or **Render** |
| Internal tool, predictable low load | **One VM** with Docker |
| Public API, traffic varies (the typical ML case) | **Cloud Run** / **Fargate** ⭐ |
| Many models, regulated data, need monitoring | **SageMaker** / **Vertex AI** |
| Sub-100 ms strict latency, high scale | **Kubernetes** with autoscaling |
| Tiny stateless function, sporadic calls | **Lambda** |

## Concepts You'll Bump Into Right Away

> **Region:** A geographic location for the cloud's data centers (e.g. `us-central1`, `ap-south-1`). Pick the one closest to your users for low latency.

> **IAM (Identity & Access Management):** Who can do what. Every cloud action requires permissions. The #1 source of "why doesn't this work" errors.

> **VPC (Virtual Private Cloud):** Your isolated network slice in the cloud. Most demos ignore this; production systems carefully wire VPCs together.

> **Cold start:** First request after the service was idle. The platform spins up a fresh container, which takes 0.5–10 s for ML images. Subsequent requests are fast.

> **Auto-scaling:** Cloud watches request rate / CPU and adds or removes copies of your service automatically.

> **Egress costs:** *Outbound* data transfer is charged. Inbound is usually free. You can rack up surprise bills sending big payloads back to clients.

> **Logs & Metrics:** Every cloud has a built-in log viewer (CloudWatch on AWS, Cloud Logging on GCP). Always look there first when something breaks.

> **Secrets:** API keys, DB passwords. Never bake them into a Docker image — use the platform's secret manager (AWS Secrets Manager, GCP Secret Manager) and inject as env vars at runtime.

## Pricing — Quick Mental Model

Cloud bills come from three buckets:

```
Compute     (CPU/RAM/GPU time)            — usually the biggest line
Storage     (disk, model files, data)     — small for ML inference
Network     (egress + load balancers)     — sneaky, can dominate at scale
```

Always set a **billing alert** before deploying anything: AWS Billing → Budgets, or GCP Billing → Budgets & alerts. A $5 alert catches forgotten resources before they become a $500 bill.

## Mental Model: From Notebook to Live URL (Cloud Run Path)

```
  Laptop                          GCP                                Internet
 ┌────────────┐    docker push   ┌─────────────────────────┐
 │ main.py    │  ───────────►    │ Container Registry      │
 │ Dockerfile │                  │  gcr.io/proj/iris-api   │
 │ model      │                  └────────────┬────────────┘
 └────────────┘                               │ gcloud run deploy
                                              ▼
                                   ┌─────────────────────────┐       https://iris-api-xxx.run.app
                                   │ Cloud Run service       │ ◄─────────────────────────────────  user
                                   │  • auto-scales 0 → N    │
                                   │  • HTTPS automatic      │
                                   │  • per-request billing  │
                                   └─────────────────────────┘
```

Three artifacts (image + tag + deploy command) → public, scalable, monitored URL. The cells below show the exact commands for two paths: **Cloud Run** (simplest), **EC2 VM** (most fundamental), and **SageMaker** (most ML-feature-rich).

In [ ]:
# EXAMPLE 1 — Google Cloud Run (serverless containers, simplest path)
# What you need: a GCP account + the `gcloud` CLI installed + a Dockerfile (Section 4).
# End result: a public HTTPS URL that auto-scales 0 → N.

cloud_run = '''
# ── Prerequisite: log in once ────────────────────────────────────────
gcloud auth login
gcloud config set project MY_PROJECT          # your GCP project id

# ── 1. Build & push image to Google Container Registry ───────────────
# `builds submit` runs docker build IN THE CLOUD (no local Docker needed!)
gcloud builds submit --tag gcr.io/MY_PROJECT/iris-api

# ── 2. Deploy ────────────────────────────────────────────────────────
# This single command:
#   • creates a Cloud Run service called "iris-api"
#   • gives it a public HTTPS URL
#   • configures auto-scaling 0 → 100 instances
#   • routes traffic with a free Google-managed SSL cert
gcloud run deploy iris-api \\
    --image gcr.io/MY_PROJECT/iris-api \\
    --platform managed \\
    --region us-central1 \\
    --allow-unauthenticated \\
    --memory 512Mi \\
    --cpu 1 \\
    --max-instances 10

# Output looks like:
#   Service URL: https://iris-api-abc123-uc.a.run.app

# ── 3. Test it from anywhere ─────────────────────────────────────────
curl -X POST https://iris-api-abc123-uc.a.run.app/predict \\
    -H "Content-Type: application/json" \\
    -d \'{"features":[5.1,3.5,1.4,0.2]}\'

# ── 4. View logs / metrics ───────────────────────────────────────────
gcloud run services logs read iris-api --region us-central1 --limit 50

# ── 5. Update with a new version (zero-downtime) ─────────────────────
gcloud builds submit --tag gcr.io/MY_PROJECT/iris-api:v2
gcloud run deploy iris-api --image gcr.io/MY_PROJECT/iris-api:v2 --region us-central1

# ── 6. Tear down (avoid surprise bills) ──────────────────────────────
gcloud run services delete iris-api --region us-central1
'''
print(cloud_run)
print('Cost on free tier: ~$0/month for low traffic. Pay only when handling requests.')

In [ ]:
# EXAMPLE 2 — AWS EC2 (rent a Linux box and run Docker on it yourself)
# This is the most "fundamental" deployment — you SEE the server, you SSH into it.
# Good for understanding what every other option is hiding from you.

ec2 = '''
# ── 1. Launch a tiny Ubuntu VM via the AWS Console ───────────────────
#   • EC2 → Launch Instance
#   • AMI:           Ubuntu Server 22.04 LTS
#   • Instance type: t3.small  (~$15/month, 2 GB RAM — enough for sklearn)
#   • Key pair:      create one, download the .pem file
#   • Security group: allow inbound TCP on port 8000 (your app) AND 22 (SSH)
#   → AWS gives you a public IP, e.g. 54.123.45.67

# ── 2. SSH in from your laptop ──────────────────────────────────────
chmod 400 my-key.pem
ssh -i my-key.pem ubuntu@54.123.45.67

# ── 3. Install Docker on the VM (one time) ──────────────────────────
sudo apt update && sudo apt install -y docker.io
sudo usermod -aG docker ubuntu          # so you can run docker without sudo
exit && ssh -i my-key.pem ubuntu@54.123.45.67   # re-login for group change

# ── 4. Push your image somewhere the VM can pull from ───────────────
#   On your LAPTOP:
docker tag iris-api:1.0 your-dockerhub-user/iris-api:1.0
docker push your-dockerhub-user/iris-api:1.0

# ── 5. Pull and run on the VM ───────────────────────────────────────
#   On the VM:
docker pull your-dockerhub-user/iris-api:1.0
docker run -d --name iris -p 8000:8000 --restart=always \\
    your-dockerhub-user/iris-api:1.0
#   --restart=always: container restarts if it crashes or VM reboots

# ── 6. Test from your laptop ────────────────────────────────────────
curl -X POST http://54.123.45.67:8000/predict \\
    -H "Content-Type: application/json" \\
    -d \'{"features":[5.1,3.5,1.4,0.2]}\'

# ── 7. Tear down ────────────────────────────────────────────────────
# AWS Console → EC2 → Instances → select → Instance state → Terminate
# !!! Forgetting this is how people get $200 surprise bills !!!
'''
print(ec2)
print('Note: no HTTPS, no autoscaling, no failover. Add those by moving up to Cloud Run / Fargate.')

In [ ]:
# EXAMPLE 3 — AWS SageMaker (managed ML platform — no Dockerfile, no Flask)
# You give SageMaker the model file + a tiny inference script.
# It builds the container, hosts it, autoscales, monitors, and gives you an HTTPS endpoint.

sagemaker = '''
# ── 1. Bundle your model: model.tar.gz containing model.joblib ──────
# Upload to S3:
aws s3 cp model.tar.gz s3://my-bucket/iris/model.tar.gz

# ── 2. Tiny inference.py SageMaker calls for you ────────────────────
# (lives next to the model; SageMaker invokes model_fn → predict_fn)
import joblib, json, os, numpy as np

def model_fn(model_dir):                   # how to load the model
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def predict_fn(input_data, model):         # how to run inference
    return model.predict(np.array(input_data["features"]).reshape(1, -1)).tolist()

# ── 3. Deploy from a notebook / Python script ───────────────────────
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

sk_model = SKLearnModel(
    model_data="s3://my-bucket/iris/model.tar.gz",
    role="arn:aws:iam::123:role/SageMakerExecutionRole",
    entry_point="inference.py",
    framework_version="1.2-1",            # which sklearn version SageMaker should use
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",          # ~$50/month if left running
    endpoint_name="iris-endpoint",
)

# ── 4. Call it ──────────────────────────────────────────────────────
predictor.predict({"features": [5.1, 3.5, 1.4, 0.2]})
# → [1]

# ── 5. SageMaker gives you for free: ────────────────────────────────
#   • Built-in CloudWatch metrics (latency, errors, invocations)
#   • Auto-scaling rules ("scale up when CPU > 70%")
#   • Multi-model endpoints (host 100 small models on one endpoint)
#   • A/B traffic splitting between model versions
#   • Model monitoring (data drift / quality drift detection)

# ── 6. !!! DELETE !!! ───────────────────────────────────────────────
predictor.delete_endpoint()
# An idle ml.t2.medium costs ~$50/month. Always tear down demos.
'''
print(sagemaker)

In [ ]:
# EXAMPLE 4 — Hugging Face Spaces (the easy free-tier path for your portfolio)
# Push a Git repo with one file → live public URL. No credit card, no Docker, no cloud account.

hf_spaces = '''
# ── 1. Sign up at https://huggingface.co (free) ─────────────────────

# ── 2. Create a new Space ───────────────────────────────────────────
#   huggingface.co → New Space → SDK = "Gradio" (or "Docker" for FastAPI)
#   This creates a Git repo for you, e.g. https://huggingface.co/spaces/yourname/iris

# ── 3. Add app.py to the repo ───────────────────────────────────────
import gradio as gr
import joblib, numpy as np

model = joblib.load("model.joblib")
CLASSES = ["setosa", "versicolor", "virginica"]

def predict(sl, sw, pl, pw):
    pred = int(model.predict(np.array([[sl, sw, pl, pw]]))[0])
    return CLASSES[pred]

gr.Interface(
    predict,
    inputs=[gr.Number(label="sepal length"), gr.Number(label="sepal width"),
            gr.Number(label="petal length"), gr.Number(label="petal width")],
    outputs="text",
    title="Iris Classifier",
).launch()

# ── 4. Add requirements.txt ─────────────────────────────────────────
# gradio
# scikit-learn
# joblib
# numpy

# ── 5. Push: model.joblib + app.py + requirements.txt ───────────────
git add model.joblib app.py requirements.txt
git commit -m "Initial deploy"
git push

# ── 6. HF rebuilds and gives you a public URL in ~2 min ─────────────
# https://huggingface.co/spaces/yourname/iris
# It has a UI for humans AND a REST API for code (auto-generated!)
'''
print(hf_spaces)
print('Best for: portfolios, sharing with non-technical stakeholders, MVPs.')
print('Cost: $0. Resource limit: 16 GB RAM, 2 CPUs (free tier).')

| | |
|--|--|
| ✅ Auto-scale to handle traffic spikes | ❌ Cloud bills can balloon if not watched |
| ✅ Public HTTPS URL out of the box | ❌ Vendor lock-in (especially with managed ML) |
| ✅ Built-in monitoring & logging | ❌ Cold starts on serverless options |

> **Reference:** [GeeksForGeeks — Cloud Computing Basics](https://www.geeksforgeeks.org/cloud-computing/) · [Google Cloud Run Quickstart](https://cloud.google.com/run/docs/quickstarts)

---
# SECTION 6: CI/CD FOR ML

## What is CI/CD? (Start Here)

**Simple Story:**
You push code at 6 PM. At 6:01 PM, a robot:
1. Runs your tests,
2. Builds your Docker image,
3. If everything passes, deploys to production — *while you go home*.

That robot is a **CI/CD pipeline**.

## Definition
> **CI — Continuous Integration:** Every push automatically runs tests, lints, and a build. Catches bugs minutes after they're written instead of weeks later.

> **CD — Continuous Delivery / Deployment:** If CI passes, the new version is automatically packaged and (optionally) released to production.

## Why ML Needs Extra Stages

A regular CI/CD pipeline only cares about **code**. ML pipelines must also handle:

```
Regular CI/CD:                ML CI/CD:
  test → build → deploy         test → train → evaluate → compare → build → deploy
                                              ▲              ▲
                                  fresh model  │              │ only deploy if
                                  on new data  │              │ new model > old model
```

Extra concerns:
1. **Data** — schema changes, new files, drift
2. **Model itself** — retrain, evaluate, only deploy if it beats the current one
3. **Reproducibility** — pin Python + library versions, log seeds, version the data
4. **Monitoring** — track prediction distributions in production

## GitHub Actions in 30 Seconds

> **GitHub Actions:** GitHub's built-in CI. Drop a YAML file at `.github/workflows/<name>.yml`; GitHub runs it on every push.

Each YAML defines:
- **`on:`** — when to trigger (push, pull_request, schedule)
- **`jobs:`** — what to run (each job runs on a fresh VM)
- **`steps:`** — sequence of shell commands or pre-built actions

## MLOps Maturity Ladder

| Level | Description |
|-------|-------------|
| 0 | Manual: notebook → scp model.joblib to server |
| 1 | Scheduled retraining script on a cron |
| 2 | Full CI/CD: tested + deployed automatically on push |
| 3 | + drift detection, automatic retraining triggers, A/B model rollouts |

In [ ]:
# A minimal but realistic GitHub Actions workflow for an ML project
workflow_yml = '''
# .github/workflows/ml-pipeline.yml
name: ML CI/CD

on:
  push:
    branches: [main]
  schedule:
    - cron: "0 2 * * 1"        # also retrain every Monday 02:00 UTC

jobs:
  build-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4                          # 1. get the code

      - uses: actions/setup-python@v5                      # 2. install Python
        with: { python-version: "3.11" }

      - name: Install deps                                  # 3. install libs
        run: pip install -r requirements.txt pytest

      - name: Run unit tests                                # 4. CI — tests
        run: pytest tests/ -v

      - name: Retrain model                                 # 5. ML — retrain
        run: python train.py --output model.joblib

      - name: Evaluate (fail if accuracy < 0.90)            # 6. ML — gate
        run: python evaluate.py --model model.joblib --min-acc 0.90

      - name: Build & push Docker image                     # 7. CD — package
        run: |
          docker build -t $REGISTRY/iris-api:${{ github.sha }} .
          docker push      $REGISTRY/iris-api:${{ github.sha }}
        env:
          REGISTRY: ${{ secrets.REGISTRY_URL }}

      - name: Deploy to Cloud Run                           # 8. CD — release
        run: gcloud run deploy iris-api \\
             --image $REGISTRY/iris-api:${{ github.sha }} \\
             --region us-central1
'''
print(workflow_yml)

In [ ]:
# evaluate.py — the script the workflow calls in step 6.
# This is the GATE that turns CI/CD into safe ML CD: a non-zero exit fails the pipeline.

evaluate_py = '''
# evaluate.py  —  exits 1 if the model is not good enough; blocks deploy.
import argparse, sys, joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

p = argparse.ArgumentParser()
p.add_argument("--model",   required=True)
p.add_argument("--min-acc", type=float, default=0.9)
args = p.parse_args()

X, y = load_iris(return_X_y=True)
_, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

acc = joblib.load(args.model).score(X_te, y_te)
print(f"Accuracy: {acc:.4f}  (threshold: {args.min_acc})")

if acc < args.min_acc:
    print("BELOW THRESHOLD — failing the pipeline.")
    sys.exit(1)            # non-zero exit = workflow step fails = deploy blocked
print("PASSED.")
'''
print(evaluate_py)

| | |
|--|--|
| ✅ No more manual deploys | ❌ Pipeline maintenance is itself work |
| ✅ Bad models can never reach production | ❌ Slow tests slow down every push |
| ✅ Reproducible — same steps every time | ❌ Secret management is critical (don't leak keys) |

> **Reference:** [GeeksForGeeks — CI/CD Pipeline](https://www.geeksforgeeks.org/what-is-ci-cd/) · [GitHub Actions Docs](https://docs.github.com/actions)

---
# SUMMARY — The Full Deployment Path

```
(1) Train model in notebook
         │
         ▼
(2) joblib.dump(model, "model.joblib")          ← Section 1: Pickle / Joblib
         │
         ▼
(3) Wrap in FastAPI    (main.py)                ← Section 2: Flask / FastAPI
         │
         ▼
(4) Design RESTful endpoints (/predict etc.)    ← Section 3: REST API
         │
         ▼
(5) Containerize with Dockerfile                ← Section 4: Docker
         │
         ▼
(6) Push image to a registry, deploy on cloud   ← Section 5: Cloud
         │
         ▼
(7) Automate the whole thing in CI/CD           ← Section 6: CI/CD
         │
         ▼
(8) Monitor predictions for drift  →  retrain  →  loop
```

Every section above is one rung on the ladder. None of them is optional in a real production system, though the *complexity* of each can be turned up or down to match team size and traffic.